# Interactive Brokers — Historical Data Pull

Connect to a locally running TWS (Trader Workstation) and pull historical bar data.
Saves to NautilusTrader's `ParquetDataCatalog` format (`data/bar/*.parquet`)
so the data is immediately available for backtesting via `BacktestDataConfig`.

## Prerequisites
- TWS installed and running with API enabled (see setup instructions below)
- IB account with market data subscriptions

## TWS Installation & Setup

1. **Download TWS** from https://www.interactivebrokers.com/en/trading/download-tws.php
2. **Install** the `.dmg` file (Mac) and launch Trader Workstation
3. **Log in** with your IB username, password, and complete 2FA via the IBKR mobile app
4. **Enable API access:**
   - Go to **Edit > Global Configuration > API > Settings**
   - Check **Enable ActiveX and Socket Clients**
   - Set **Socket port** to `7496` (live) or `7497` (paper)
   - Uncheck **Read-Only API** if you want order execution (leave checked for data-only)
   - Click **Apply** and **OK**
5. **Keep TWS running** while using this notebook

## Step 0: Check TWS is reachable

Run this first to verify TWS is running and accepting API connections.
No imports or setup needed — just a raw socket check.

In [1]:
import socket

_host = "127.0.0.1"
_port = 7496  # 7496 = live, 7497 = paper

try:
    with socket.create_connection((_host, _port), timeout=3):
        print(f"OK — TWS is accepting connections at {_host}:{_port}")
except ConnectionRefusedError:
    print(f"BLOCKED — Connection refused at {_host}:{_port}")
    print()
    print("TWS is not accepting API connections. Check:")
    print("  1. Is TWS running and are you logged in?")
    print("  2. Is API enabled? Edit > Global Configuration > API > Settings")
    print("     - 'Enable ActiveX and Socket Clients' must be checked")
    print(f"     - Socket port must be {_port}")
    print("  3. If using paper trading, change _port to 7497 above")
except TimeoutError:
    print(f"BLOCKED — Connection timed out at {_host}:{_port}")
    print()
    print("TWS may be rate-limiting or not responding.")
    print("Wait a few minutes and try again.")
except OSError as e:
    print(f"BLOCKED — {e}")
    print()
    print("Check that TWS is running and your network is working.")

OK — TWS is accepting connections at 127.0.0.1:7496


In [ ]:
import asyncio
import datetime
import os
import socket
import time
from pathlib import Path

from nautilus_trader.adapters.interactive_brokers.common import IBContract
from nautilus_trader.adapters.interactive_brokers.historical.client import (
    HistoricInteractiveBrokersClient,
)
from nautilus_trader.persistence.catalog import ParquetDataCatalog

from ib_rate_limiter import IBRateLimiter

# === TWS Connection Settings ===
TWS_HOST = "127.0.0.1"
TWS_PORT = 7496  # 7496 = live, 7497 = paper
TWS_CLIENT_ID = 1

# === Catalog Settings ===
# This must match the NAUTILUS_STORE_PATH used by the Automatron server
# Read from NAUTILUS_STORE_PATH env var (same as the server), default to project root
CATALOG_PATH = Path(
    os.environ.get("NAUTILUS_STORE_PATH", "/Users/mordrax/code/nautilus_automatron/backtest_catalog")
)


# === IB Rate Limits ===
# Max duration per single reqHistoricalData call, keyed by bar size in seconds.
# Value is (timedelta for chunking, IB duration string for the API call).
_MAX_CHUNK = {
    5:     (datetime.timedelta(hours=1),   "3600 S"),
    10:    (datetime.timedelta(hours=4),   "14400 S"),
    15:    (datetime.timedelta(hours=4),   "14400 S"),
    30:    (datetime.timedelta(hours=8),   "28800 S"),
    60:    (datetime.timedelta(days=1),    "1 D"),
    120:   (datetime.timedelta(days=2),    "2 D"),
    180:   (datetime.timedelta(weeks=1),   "1 W"),
    300:   (datetime.timedelta(weeks=1),   "1 W"),
    600:   (datetime.timedelta(weeks=1),   "1 W"),
    900:   (datetime.timedelta(weeks=1),   "1 W"),
    1200:  (datetime.timedelta(weeks=1),   "1 W"),
    1800:  (datetime.timedelta(weeks=1),   "1 W"),
    3600:  (datetime.timedelta(days=30),   "1 M"),
    86400: (datetime.timedelta(days=365),  "1 Y"),
}

_BAR_SPEC_TO_SECONDS = {
    "5-SECOND": 5, "10-SECOND": 10, "15-SECOND": 15, "30-SECOND": 30,
    "1-MINUTE": 60, "2-MINUTE": 120, "3-MINUTE": 180, "5-MINUTE": 300,
    "10-MINUTE": 600, "15-MINUTE": 900, "20-MINUTE": 1200, "30-MINUTE": 1800,
    "1-HOUR": 3600, "1-DAY": 86400,
}

def _parse_bar_seconds(bar_spec: str) -> int:
    """Parse '1-MINUTE-MID' -> 60 (seconds for the size-timeframe part)."""
    parts = bar_spec.rsplit("-", 1)  # split off price type
    size_tf = parts[0]  # e.g. '1-MINUTE'
    if size_tf not in _BAR_SPEC_TO_SECONDS:
        raise ValueError(f"Unknown bar specification: {bar_spec}. Known: {list(_BAR_SPEC_TO_SECONDS.keys())}")
    return _BAR_SPEC_TO_SECONDS[size_tf]


def _chunk_date_range(start: datetime.datetime, end: datetime.datetime, chunk_size: datetime.timedelta):
    """Yield (chunk_start, chunk_end) pairs walking backward from end to start."""
    current_end = end
    while current_end > start:
        chunk_start = max(current_end - chunk_size, start)
        yield chunk_start, current_end
        current_end = chunk_start


# Global rate limiter — shared across all pulls in this session
rate_limiter = IBRateLimiter()


async def _request_with_timeout(client, bar_spec, chunk_end, chunk_duration, contract, use_rth, timeout):
    """Wrap client.request_bars with asyncio.wait_for to enforce a hard timeout."""
    return await asyncio.wait_for(
        client.request_bars(
            bar_specifications=[bar_spec],
            end_date_time=chunk_end,
            duration=chunk_duration,
            tz_name="UTC",
            contracts=[contract],
            use_rth=use_rth,
            timeout=timeout,
        ),
        timeout=timeout + 5,  # outer timeout slightly longer than inner
    )


def _validate_bars(bars, chunk_start, chunk_end):
    """Validate returned bars. Returns (valid_bars, issues)."""
    if not bars:
        return [], []
    issues = []

    # Check timestamps are within expected range (with 1-day tolerance for IB quirks)
    tolerance = datetime.timedelta(days=1)
    first_ts = bars[0].ts_event
    last_ts = bars[-1].ts_event

    expected_start_ns = int(
        (chunk_start - tolerance).replace(tzinfo=datetime.timezone.utc).timestamp() * 1e9
    )
    expected_end_ns = int(
        (chunk_end + tolerance).replace(tzinfo=datetime.timezone.utc).timestamp() * 1e9
    )

    if first_ts < expected_start_ns:
        issues.append(f"first bar timestamp before expected range")
    if last_ts > expected_end_ns:
        issues.append(f"last bar timestamp after expected range")

    # Check for zero/negative prices
    bad_prices = sum(1 for b in bars if float(b.open) <= 0 or float(b.close) <= 0)
    if bad_prices > 0:
        issues.append(f"{bad_prices} bars with zero/negative prices")

    return bars, issues


async def pull_bars(
    client,
    contract: IBContract,
    bar_spec: str,
    start: datetime.datetime,
    end: datetime.datetime,
    use_rth: bool = False,
    timeout_per_chunk: int = 10,
    max_retries: int = 3,
    retry_delay: float = 5.0,
) -> list:
    """Pull historical bars with rate limiting, retries, and data validation.

    Splits the date range into chunks based on IB's max duration per bar size,
    paces requests to stay within IB's rate limits, retries on failure, and
    validates each chunk's data.
    """
    bar_seconds = _parse_bar_seconds(bar_spec)
    chunk_td, duration_str = _MAX_CHUNK[bar_seconds]

    chunks = list(_chunk_date_range(start, end, chunk_td))
    total_chunks = len(chunks)

    print(f"[pull] {contract.symbol} {bar_spec}")
    print(f"[pull] Range: {start.date()} to {end.date()} ({(end - start).days} days)")
    print(f"[pull] Chunk size: {chunk_td} (duration: {duration_str}), {total_chunks} requests needed")
    print(f"[pull] Rate limiter: {rate_limiter.remaining} requests remaining in window")
    print(f"[pull] Timeout: {timeout_per_chunk}s per chunk, {max_retries} retries")
    if total_chunks > rate_limiter.remaining:
        est_pauses = (total_chunks - rate_limiter.remaining) // rate_limiter.max_requests
        print(f"[pull] NOTE: Will need to pause for rate limiting (~{est_pauses * 10} min of waiting)")
    print()

    # Ensure instrument is loaded (required before requesting bars)
    try:
        await client.request_instruments(contracts=[contract])
    except Exception:
        pass  # may already be cached

    all_bars = []
    failed_chunks = []
    t0 = time.perf_counter()

    for i, (chunk_start, chunk_end) in enumerate(chunks):
        # Calculate actual duration for this chunk (last chunk may be shorter)
        actual_td = chunk_end - chunk_start
        if actual_td < chunk_td:
            total_secs = int(actual_td.total_seconds())
            if total_secs >= 86400:
                chunk_duration = f"{total_secs // 86400} D"
            else:
                chunk_duration = f"{max(30, total_secs)} S"
        else:
            chunk_duration = duration_str

        bars = None
        last_error = None

        for attempt in range(1, max_retries + 1):
            # Wait for rate limit slot
            await rate_limiter.acquire()

            elapsed = time.perf_counter() - t0
            retry_label = f" (retry {attempt}/{max_retries})" if attempt > 1 else ""
            print(f"  [{i + 1}/{total_chunks}] {chunk_start.date()} to {chunk_end.date()} "
                  f"dur={chunk_duration} slots={rate_limiter.remaining} t={elapsed:.0f}s{retry_label}")

            try:
                bars = await _request_with_timeout(
                    client, bar_spec, chunk_end, chunk_duration,
                    contract, use_rth, timeout_per_chunk,
                )

                if bars:
                    valid_bars, issues = _validate_bars(bars, chunk_start, chunk_end)
                    if issues:
                        print(f"         -> {len(bars)} bars (WARNINGS: {'; '.join(issues)})")
                    else:
                        print(f"         -> {len(bars)} bars")
                    all_bars.extend(valid_bars)
                    break  # success
                else:
                    print(f"         -> 0 bars (empty response)")
                    # Empty response is not necessarily an error — could be weekend/holiday
                    break

            except (asyncio.TimeoutError, asyncio.CancelledError):
                last_error = "timeout"
                print(f"         -> TIMEOUT after {timeout_per_chunk}s")
                if attempt < max_retries:
                    wait = retry_delay * attempt
                    print(f"         Retrying in {wait:.0f}s...")
                    await asyncio.sleep(wait)
            except Exception as e:
                last_error = str(e)
                print(f"         -> ERROR: {e}")
                if attempt < max_retries:
                    wait = retry_delay * attempt
                    print(f"         Retrying in {wait:.0f}s...")
                    await asyncio.sleep(wait)

        if bars is None and last_error:
            failed_chunks.append((chunk_start.date(), chunk_end.date(), last_error))
            print(f"         GIVING UP on this chunk after {max_retries} attempts")

    total_time = time.perf_counter() - t0
    print()
    print(f"[pull] Done: {len(all_bars)} total bars in {total_time:.1f}s")

    if failed_chunks:
        print(f"[pull] FAILED CHUNKS ({len(failed_chunks)}):")
        for cs, ce, err in failed_chunks:
            print(f"  {cs} to {ce}: {err}")

    # Sort by timestamp and deduplicate
    if all_bars:
        all_bars.sort(key=lambda b: b.ts_event)
        seen = set()
        deduped = []
        for bar in all_bars:
            if bar.ts_event not in seen:
                seen.add(bar.ts_event)
                deduped.append(bar)
        if len(deduped) < len(all_bars):
            print(f"[pull] Removed {len(all_bars) - len(deduped)} duplicate bars")
        all_bars = deduped

    return all_bars


def check_tws_connection(host: str, port: int) -> bool:
    """Check if TWS is accepting connections on the given host:port."""
    try:
        with socket.create_connection((host, port), timeout=3):
            return True
    except (ConnectionRefusedError, TimeoutError, OSError):
        return False


def save_bars_to_catalog(bars: list, catalog_path: Path) -> None:
    """Save bars to the NautilusTrader ParquetDataCatalog.

    Writes parquet files into:
        catalog_path/data/bar/{bar_type}/*.parquet

    Uses skip_disjoint_check=True so re-pulling overlapping date ranges
    appends a new parquet file rather than raising an error.
    """
    if not bars:
        print("WARNING: No bars to save — skipping")
        return

    bar_count = len(bars)
    bar_type_str = str(bars[0].bar_type)
    print(f"[save] {bar_count} bars of type {bar_type_str}")

    t0 = time.perf_counter()
    catalog = ParquetDataCatalog(str(catalog_path))
    catalog.write_data(bars, skip_disjoint_check=True)
    elapsed = time.perf_counter() - t0

    print(f"[save] Written to catalog in {elapsed:.2f}s")
    print(f"[save] Path: {catalog_path / 'data' / 'bar'}")


# --- Startup checks ---
if check_tws_connection(TWS_HOST, TWS_PORT):
    print(f"TWS API detected at {TWS_HOST}:{TWS_PORT}")
else:
    print(f"WARNING: Cannot reach TWS at {TWS_HOST}:{TWS_PORT}")
    print()
    print("Please ensure:")
    print("  1. TWS (Trader Workstation) is running and you are logged in")
    print("  2. API is enabled: Edit > Global Configuration > API > Settings")
    print("     - 'Enable ActiveX and Socket Clients' is checked")
    print(f"     - Socket port is set to {TWS_PORT}")
    print("  3. If using paper trading, change TWS_PORT to 7497 above")
    print()
    print("Download TWS: https://www.interactivebrokers.com/en/trading/download-tws.php")

print(f"Catalog path: {CATALOG_PATH} ({'exists' if CATALOG_PATH.exists() else 'MISSING'})")

## 1. Connect Historical Data Client

Connects to your locally running TWS for historical data requests.
This cell is safe to re-run — it reuses the existing client if already connected.

> **Note:** If the connection fails, check the warning output from the cell above.

In [3]:
import asyncio
import subprocess
import re

def _kill_stale_ib_connections(current_pid: int, tws_port: int = 7496):
    """Find and kill orphaned Python processes holding IB connections on tws_port."""
    try:
        result = subprocess.run(
            ["lsof", "-i", f":{tws_port}", "-P"],
            capture_output=True, text=True, timeout=5,
        )
        stale_pids = set()
        for line in result.stdout.splitlines()[1:]:
            parts = line.split()
            if len(parts) >= 2 and parts[0] == "Python":
                pid = int(parts[1])
                if pid != current_pid:
                    stale_pids.add(pid)

        if stale_pids:
            print(f"  Found {len(stale_pids)} stale process(es) holding port {tws_port}: {stale_pids}")
            for pid in stale_pids:
                try:
                    import os
                    os.kill(pid, 15)
                    print(f"  Killed PID {pid}")
                except ProcessLookupError:
                    pass
            import time
            time.sleep(1)
        else:
            print(f"  No stale connections on port {tws_port}")
    except Exception as e:
        print(f"  WARNING: Could not check for stale connections: {e}")


def _get_current_kernel_pid() -> int:
    """Get the PID of the current Jupyter kernel process."""
    import os
    return os.getpid()


# --- Clean up stale connections ---
print("Checking for stale IB connections...")
_kill_stale_ib_connections(
    current_pid=_get_current_kernel_pid(),
    tws_port=TWS_PORT,
)
print()

# --- Connect ---
if "client" in dir() and client._client.is_running:
    print(f"Already connected to TWS at {TWS_HOST}:{TWS_PORT}")
else:
    if not check_tws_connection(TWS_HOST, TWS_PORT):
        raise ConnectionError(
            f"Cannot reach TWS at {TWS_HOST}:{TWS_PORT}. "
            "Make sure TWS is running and API is enabled."
        )

    try:
        client = HistoricInteractiveBrokersClient(
            host=TWS_HOST,
            port=TWS_PORT,
            client_id=TWS_CLIENT_ID,
        )
    except RuntimeError as e:
        if "Logging subsystem already initialized" in str(e):
            import nautilus_trader.common.component as _ntc
            _original_init = _ntc.init_logging
            _ntc.init_logging = lambda **kwargs: None
            try:
                client = HistoricInteractiveBrokersClient(
                    host=TWS_HOST,
                    port=TWS_PORT,
                    client_id=TWS_CLIENT_ID,
                )
            finally:
                _ntc.init_logging = _original_init
        else:
            raise

    await client.connect()
    await asyncio.sleep(2)  # REQUIRED: wait for IB to be ready before making requests
    print(f"Connected to TWS at {TWS_HOST}:{TWS_PORT}")

Checking for stale IB connections...
  Found 1 stale process(es) holding port 7496: {62076}
  Killed PID 62076

2026-03-27T05:30:37.712413000Z [INFO] TRADER-000.MessageBus: config.database=None
2026-03-27T05:30:37.712420000Z [INFO] TRADER-000.MessageBus: config.encoding='msgpack'
2026-03-27T05:30:37.712421000Z [INFO] TRADER-000.MessageBus: config.timestamps_as_iso8601=False
2026-03-27T05:30:37.712423000Z [INFO] TRADER-000.MessageBus: config.buffer_interval_ms=None
2026-03-27T05:30:37.712424000Z [INFO] TRADER-000.MessageBus: config.autotrim_mins=None
2026-03-27T05:30:37.712424001Z [INFO] TRADER-000.MessageBus: config.use_trader_prefix=True
2026-03-27T05:30:37.712424002Z [INFO] TRADER-000.MessageBus: config.use_trader_id=True
2026-03-27T05:30:37.712424003Z [INFO] TRADER-000.MessageBus: config.use_instance_id=False
2026-03-27T05:30:37.712425000Z [INFO] TRADER-000.MessageBus: config.streams_prefix='stream'
2026-03-27T05:30:37.712425001Z [INFO] TRADER-000.MessageBus: config.types_filter=Non

## 2. Pull Historical Data

Configure the instrument, bar size, and date range below.
The `pull_bars` function handles chunking and rate limiting automatically:

- Splits requests by IB's max duration per bar size (e.g., 1 day for 1-min bars, 1 week for 5-min)
- Stays within 55 requests per 10-minute window (IB limit is 60)
- Enforces minimum 3s delay between requests (IB limit is 6 per 2 seconds)
- Shows progress and pauses automatically when nearing the limit

### Available Bar Specifications
Format: `{size}-{timeframe}-{price_type}` where price_type is one of:
- `MID` — midpoint (use this for CFDs like XAUUSD)
- `LAST` — last trade price (use for stocks, futures)
- `BID` / `ASK` — bid or ask price

### Max duration per bar size (handled automatically)
| Bar size | Max per request | 30 days = N requests |
|----------|----------------|---------------------|
| 1-MINUTE | 1 day | 30 |
| 5-MINUTE | 1 week | 5 |
| 1-HOUR | 1 month | 1 |
| 1-DAY | 1 year | 1 |

In [4]:
# XAUUSD (Spot Gold CFD)
xauusd_contract = IBContract(
    conId=457068913,
    secType="CFD",
    symbol="XAUUSD",
    exchange="SMART",
    currency="USD",
)

# Request instrument first (required for bars to work)
xauusd_instruments = await client.request_instruments(contracts=[xauusd_contract])
print(f"Instrument: {xauusd_instruments[0].id if xauusd_instruments else 'FAILED'}")

# Date range
end_date = datetime.datetime.now()
start_date = end_date - datetime.timedelta(days=30)

print(f"Date range: {start_date.date()} to {end_date.date()}")

Instrument: XAUUSD.IBCFD2026-03-27T05:37:36.446265000Z [INFO] TRADER-000.InteractiveBrokersInstrumentProvider: Contract qualified for XAUUSD.SMART with ConId=457068913

Date range: 2026-02-25 to 2026-03-27
2026-03-27T05:37:36.446743000Z [INFO] TRADER-000.InteractiveBrokersInstrumentProvider: Adding instrument=Cfd(id=XAUUSD.IBCFD, raw_symbol=XAUUSD, asset_class=INDEX, instrument_class=CFD, quote_currency=USD, is_inverse=False, price_precision=2, price_increment=0.01, size_precision=0, size_increment=1, multiplier=1, lot_size=None, margin_init=0, margin_maint=0, maker_fee=0, taker_fee=0, info={'contract': {'secType': 'CFD', 'conId': 457068913, 'exchange': 'SMART', 'primaryExchange': '', 'symbol': 'XAUUSD', 'localSymbol': 'XAUUSD', 'currency': 'USD', 'tradingClass': 'XAUUSD', 'lastTradeDateOrContractMonth': '', 'lastTradeDate': '', 'multiplier': '', 'strike': 1.7976931348623157e+308, 'right': '', 'includeExpired': False, 'secIdType': '', 'secId': '', 'description': '', 'issuerId': '', 'co

### Example 1: Pull 1-Minute Bars

In [5]:
if "client" not in dir() or not client._client.is_running:
    print("ERROR: Not connected to TWS. Run the connection cell above first.")
else:
    bars_1m = await pull_bars(
        client, xauusd_contract, "1-MINUTE-MID",
        start=start_date, end=end_date,
    )

[pull] XAUUSD 1-MINUTE-MID
[pull] Range: 2026-02-25 to 2026-03-27 (30 days)
[pull] Chunk size: 1 day, 0:00:00 (duration: 1 D), 30 requests needed
[pull] Rate limiter: 55 requests remaining in window
[pull] Timeout: 10s per chunk, 3 retries

  [1/30] 2026-03-26 to 2026-03-27 dur=1 D slots=54 t=0s
2026-03-27T05:37:47.600625000Z [INFO] TRADER-000.InteractiveBrokersInstrumentProvider: Contract qualified for XAUUSD.SMART with ConId=457068913
2026-03-27T05:37:47.600907000Z [INFO] TRADER-000.InteractiveBrokersInstrumentProvider: Adding instrument=Cfd(id=XAUUSD.IBCFD, raw_symbol=XAUUSD, asset_class=INDEX, instrument_class=CFD, quote_currency=USD, is_inverse=False, price_precision=2, price_increment=0.01, size_precision=0, size_increment=1, multiplier=1, lot_size=None, margin_init=0, margin_maint=0, maker_fee=0, taker_fee=0, info={'contract': {'secType': 'CFD', 'conId': 457068913, 'exchange': 'SMART', 'primaryExchange': '', 'symbol': 'XAUUSD', 'localSymbol': 'XAUUSD', 'currency': 'USD', 'tradin

In [ ]:
if "bars_1m" in dir() and bars_1m:
    save_bars_to_catalog(bars_1m, CATALOG_PATH)
else:
    print("No bars to save — run the pull cell above first")

### Example 2: Pull 5-Minute Bars

In [ ]:
if "client" not in dir() or not client._client.is_running:
    print("ERROR: Not connected to TWS. Run the connection cell above first.")
else:
    bars_5m = await pull_bars(
        client, xauusd_contract, "5-MINUTE-MID",
        start=start_date, end=end_date,
    )

In [ ]:
if "bars_5m" in dir() and bars_5m:
    save_bars_to_catalog(bars_5m, CATALOG_PATH)
else:
    print("No bars to save — run the pull cell above first")

### Custom Pull

Modify the contract, bar spec, and date range below to pull any instrument.

In [ ]:
if "client" not in dir() or not client._client.is_running:
    print("ERROR: Not connected to TWS. Run the connection cell above first.")
else:
    # === CONFIGURE YOUR PULL HERE ===
    # secType guide:
    #   CMDTY  — commodities (XAUUSD, XAGUSD). Use MID/BID/ASK, NOT LAST.
    #   STK    — stocks (AAPL, MSFT). Use LAST.
    #   FUT    — futures (GC, ES). Use LAST.
    #   CFD    — DO NOT USE for historical data (IB limitation).
    custom_contract = IBContract(
        secType="CMDTY",
        symbol="XAUUSD",
        exchange="SMART",
        currency="USD",
    )

    custom_bar_spec = "1-MINUTE-MID"  # MID for CMDTY, LAST for STK/FUT
    custom_start = datetime.datetime.now() - datetime.timedelta(days=30)
    custom_end = datetime.datetime.now()
    # ================================

    custom_bars = await pull_bars(
        client, custom_contract, custom_bar_spec,
        start=custom_start, end=custom_end,
    )

    if custom_bars:
        save_bars_to_catalog(custom_bars, CATALOG_PATH)
    else:
        print("No bars to save")